In [0]:
# ================================================================
# CELL 1 — LOAD DEPENDENCIES
# ================================================================


In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/02_question_classifier.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/03_sql_generator.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/04_sql_validator.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/05_sql_executor.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/04_rag_retrieval.py

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/07_rag/05_rag_answer_generation.py

In [0]:
# ================================================================
# CELL 2 — IMPORTS AND CONFIGURATION
# ================================================================

import json
import time
import inspect

from pyspark.sql import functions as F


CATALOG = "genai_copilot"

SILVER_TABLE = "genai_copilot.silver.sales"

GOLD_TABLE = "genai_copilot.gold.region_sales"


print("=" * 70)
print("COPILOT ORCHESTRATOR")
print("=" * 70)

print()
print("Catalog:", CATALOG)
print("Silver table:", SILVER_TABLE)
print("Gold table:", GOLD_TABLE)

In [0]:
# ================================================================
# CELL 3 — DEPENDENCY VERIFICATION
# ================================================================

required_functions = [
    "classify_question",
    "generate_sql_request",
    "validate_sql",
    "execute_sql",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer"
]


print("=" * 70)
print("DEPENDENCY CHECK")
print("=" * 70)


failed_dependencies = []


for function_name in required_functions:

    function_object = globals().get(
        function_name
    )

    available = callable(
        function_object
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:

        failed_dependencies.append(
            function_name
        )


print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)


if failed_dependencies:

    raise RuntimeError(
        "Missing required functions: "
        + ", ".join(
            failed_dependencies
        )
    )


# ================================================================
# VERIFY RAG FUNCTION
# ================================================================

print()
print("=" * 70)
print("RAG ANSWER GENERATOR CHECK")
print("=" * 70)


rag_source = inspect.getsource(
    generate_rag_answer
)


rag_source_lower = rag_source.lower()


if "dummy answer" in rag_source_lower:

    raise RuntimeError(
        "ERROR: Dummy generate_rag_answer() detected."
    )


print(
    "generate_rag_answer:",
    generate_rag_answer
)

print()
print(
    "Source file:",
    inspect.getsourcefile(
        generate_rag_answer
    )
)

print()
print(
    "RAG implementation: PASS"
)

print(
    "Dummy implementation: NOT DETECTED"
)

In [0]:
# ================================================================
# CELL 4 — ROUTE DETERMINATION
# ================================================================

def determine_route(classification):
    """
    Determine whether the question should use
    SQL, RAG, Hybrid, or Unsupported processing.
    """

    if not isinstance(
        classification,
        dict
    ):

        return "unsupported"


    if classification.get(
        "requires_hybrid",
        False
    ):

        return "hybrid"


    if (
        classification.get(
            "requires_sql",
            False
        )
        and
        classification.get(
            "requires_rag",
            False
        )
    ):

        return "hybrid"


    if classification.get(
        "requires_sql",
        False
    ):

        return "sql"


    if classification.get(
        "requires_rag",
        False
    ):

        return "rag"


    intent = str(
        classification.get(
            "intent",
            ""
        )
    ).lower()


    if intent == "unsupported":

        return "unsupported"


    return "unsupported"


print(
    "determine_route(): PASS"
)

In [0]:
# ================================================================
# CELL 5 — SQL ROUTE
# ================================================================

def run_sql_route(question):
    """
    Execute the complete SQL pipeline.
    """

    start_time = time.time()

    response = {
        "success": False,
        "question": question,
        "route": "sql",
        "answer": None,
        "sql": None,
        "data": None,
        "row_count": 0,
        "error": None,
        "execution_time_ms": None
    }


    try:

        # --------------------------------------------------------
        # 1. Generate SQL
        # --------------------------------------------------------

        sql_result = generate_sql_request(
            question
        )


        if not sql_result.get(
            "success",
            False
        ):

            response["error"] = sql_result.get(
                "error",
                "SQL generation failed."
            )

            return response


        sql = sql_result.get(
            "sql"
        )


        response["sql"] = sql


        # --------------------------------------------------------
        # 2. Validate SQL
        # --------------------------------------------------------

        validation_result = validate_sql(
            sql
        )


        if not validation_result.get(
            "valid",
            False
        ):

            response["error"] = validation_result.get(
                "error",
                "SQL validation failed."
            )

            return response


        # --------------------------------------------------------
        # 3. Execute SQL
        # --------------------------------------------------------

        execution_result = execute_sql(
            sql
        )


        response["success"] = execution_result.get(
            "success",
            False
        )


        response["row_count"] = execution_result.get(
            "row_count",
            0
        )


        response["error"] = execution_result.get(
            "error"
        )


        if response["success"]:

            response["data"] = execution_result.get(
                "dataframe"
            )


        return response


    except Exception as e:

        response["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return response


    finally:

        response["execution_time_ms"] = round(
            (
                time.time()
                - start_time
            ) * 1000,
            2
        )


print(
    "run_sql_route(): PASS"
)

In [0]:
# ================================================================
# CELL 6 — HYBRID QUESTION DECOMPOSITION
# ================================================================

def decompose_hybrid_question(question):
    """
    Split a hybrid question into SQL and RAG sub-questions.
    """

    if not question:

        return {
            "success": False,
            "original_question": question,
            "sql_question": None,
            "rag_question": None,
            "decomposition_type": None,
            "error": "Question is empty."
        }


    normalized = question.strip().lower()


    # ------------------------------------------------------------
    # Revenue + Discount Policy
    # ------------------------------------------------------------

    if (
        "highest revenue" in normalized
        and "discount policy" in normalized
    ):

        return {
            "success": True,
            "original_question": question,
            "sql_question": (
                "Which region generated "
                "the highest revenue?"
            ),
            "rag_question": (
                "What discount policy applies "
                "to that region?"
            ),
            "decomposition_type": "sql_rag"
        }


    return {
        "success": False,
        "original_question": question,
        "sql_question": None,
        "rag_question": None,
        "decomposition_type": None,
        "error": (
            "Unable to decompose this "
            "hybrid question."
        )
    }


print(
    "decompose_hybrid_question(): PASS"
)

In [0]:
# ================================================================
# CELL 7 — HYBRID ROUTE
# ================================================================

def run_hybrid_route(question):
    """
    Execute SQL + RAG hybrid processing.
    """

    start_time = time.time()

    response = {
        "success": False,
        "question": question,
        "route": "hybrid",

        "sql_question": None,
        "rag_question": None,

        "sql_result": None,
        "rag_result": None,
        "rag_sources": [],

        "answer": None,
        "error": None,

        "execution_time_ms": None
    }


    try:

        # ========================================================
        # 1. DECOMPOSE
        # ========================================================

        decomposition = decompose_hybrid_question(
            question
        )


        if not decomposition.get(
            "success",
            False
        ):

            response["error"] = decomposition.get(
                "error",
                "Hybrid decomposition failed."
            )

            return response


        sql_question = decomposition.get(
            "sql_question"
        )

        rag_question = decomposition.get(
            "rag_question"
        )


        response["sql_question"] = sql_question
        response["rag_question"] = rag_question


        # ========================================================
        # 2. SQL PIPELINE
        # ========================================================

        sql_result = run_sql_route(
            sql_question
        )


        response["sql_result"] = sql_result


        if not sql_result.get(
            "success",
            False
        ):

            response["error"] = (
                "SQL pipeline failed: "
                + str(
                    sql_result.get(
                        "error"
                    )
                )
            )

            return response


        # ========================================================
        # 3. RAG EMBEDDING
        # ========================================================

        question_embedding = generate_question_embedding(
            rag_question
        )


        if question_embedding is None:

            response["error"] = (
                "Question embedding generation failed."
            )

            return response


        # ========================================================
        # 4. RAG RETRIEVAL
        # ========================================================

        retrieval_results = retrieve_documents(
            question_embedding
        )


        if retrieval_results is None:

            response["error"] = (
                "RAG retrieval returned None."
            )

            return response


        # ========================================================
        # 5. RAG SOURCES
        # ========================================================

        response["rag_sources"] = [

            {
                "chunk_id": item.get(
                    "chunk_id"
                ),

                "document_id": item.get(
                    "document_id"
                ),

                "file_name": item.get(
                    "file_name"
                ),

                "title": item.get(
                    "title"
                )
            }

            for item in retrieval_results
        ]


        # ========================================================
        # 6. BUILD RAG CONTEXT
        # ========================================================

        rag_context = build_rag_context(
            retrieval_results
        )


        if not rag_context:

            response["error"] = (
                "RAG context is empty."
            )

            return response


        # ========================================================
        # 7. GENERATE RAG ANSWER
        # ========================================================

        rag_result = generate_rag_answer(
            rag_question,
            rag_context
        )


        if not isinstance(
            rag_result,
            dict
        ):

            response["error"] = (
                "RAG answer generator returned "
                f"{type(rag_result).__name__} "
                "instead of a dictionary."
            )

            return response


        response["rag_result"] = rag_result


        # ========================================================
        # 8. CHECK RAG SUCCESS
        # ========================================================

        if not rag_result.get(
            "success",
            False
        ):

            response["error"] = (
                "RAG pipeline failed: "
                + str(
                    rag_result.get(
                        "error"
                    )
                )
            )

            return response


        # ========================================================
        # 9. HYBRID SUCCESS
        # ========================================================

        response["success"] = True


        response["answer"] = {

            "sql_data": sql_result.get(
                "data"
            ),

            "rag_answer": rag_result,

            "rag_sources": response[
                "rag_sources"
            ]
        }


        return response


    except Exception as e:

        response["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return response


    finally:

        response["execution_time_ms"] = round(
            (
                time.time()
                - start_time
            ) * 1000,
            2
        )


print(
    "run_hybrid_route(): PASS"
)

In [0]:
# ================================================================
# CELL 8 — HYBRID FINAL ANSWER ASSEMBLY
# ================================================================

def assemble_hybrid_answer(
    question,
    hybrid_result
):
    """
    Combine SQL and RAG results into
    one structured final answer.
    """

    if not hybrid_result.get(
        "success",
        False
    ):

        return {
            "success": False,
            "question": question,
            "answer": None,
            "error": hybrid_result.get(
                "error",
                "Hybrid processing failed."
            )
        }


    sql_data = hybrid_result[
        "sql_result"
    ].get(
        "data"
    )


    rag_result = hybrid_result.get(
        "rag_result"
    )


    # ------------------------------------------------------------
    # Extract RAG text
    # ------------------------------------------------------------

    if isinstance(
        rag_result,
        dict
    ):

        rag_answer = rag_result.get(
            "answer"
        )

    else:

        rag_answer = rag_result


    # ------------------------------------------------------------
    # Extract highest revenue region
    # ------------------------------------------------------------

    highest_region = None
    highest_revenue = None


    if sql_data is not None:

        rows = sql_data.collect()


        if rows:

            highest_region = rows[0][
                "region"
            ]

            highest_revenue = rows[0][
                "total_revenue"
            ]


    # ------------------------------------------------------------
    # Final answer
    # ------------------------------------------------------------

    answer = {

        "question": question,

        "highest_revenue_region": (
            highest_region
        ),

        "highest_revenue": (
            highest_revenue
        ),

        "discount_policy": (
            rag_answer
        )
    }


    return {

        "success": True,

        "question": question,

        "answer": answer,

        "error": None
    }


print(
    "assemble_hybrid_answer(): PASS"
)

In [0]:
# ================================================================
# CELL 9 — MAIN ASK_COPILOT ORCHESTRATOR
# ================================================================

def ask_copilot(question):
    """
    Main entry point for the GenAI Data Analyst Copilot.

    Routes the question to:

    - SQL
    - RAG
    - Hybrid
    - Unsupported
    """

    start_time = time.time()


    response = {

        "success": False,

        "question": question,

        "route": None,

        "answer": None,

        "sql": None,

        "data": None,

        "sources": [],

        "error": None,

        "execution_time_ms": None
    }


    try:

        # ========================================================
        # 1. CLASSIFICATION
        # ========================================================

        classification = classify_question(
            question
        )


        route = determine_route(
            classification
        )


        response["route"] = route


        # ========================================================
        # 2. SQL ROUTE
        # ========================================================

        if route == "sql":

            sql_result = run_sql_route(
                question
            )


            response["success"] = sql_result.get(
                "success",
                False
            )


            response["sql"] = sql_result.get(
                "sql"
            )


            response["data"] = sql_result.get(
                "data"
            )


            response["error"] = sql_result.get(
                "error"
            )


            if response["success"]:

                response["answer"] = (
                    "SQL analysis completed successfully."
                )


        # ========================================================
        # 3. RAG ROUTE
        # ========================================================

        elif route == "rag":

            # ----------------------------------------------------
            # Embedding
            # ----------------------------------------------------

            question_embedding = (
                generate_question_embedding(
                    question
                )
            )


            if question_embedding is None:

                response["error"] = (
                    "Question embedding generation failed."
                )

                return response


            # ----------------------------------------------------
            # Retrieval
            # ----------------------------------------------------

            retrieval_results = retrieve_documents(
                question_embedding
            )


            if retrieval_results is None:

                response["error"] = (
                    "RAG retrieval returned None."
                )

                return response


            # ----------------------------------------------------
            # Sources
            # ----------------------------------------------------

            response["sources"] = [

                {
                    "chunk_id": item.get(
                        "chunk_id"
                    ),

                    "document_id": item.get(
                        "document_id"
                    ),

                    "file_name": item.get(
                        "file_name"
                    ),

                    "title": item.get(
                        "title"
                    )
                }

                for item in retrieval_results
            ]


            # ----------------------------------------------------
            # Context
            # ----------------------------------------------------

            rag_context = build_rag_context(
                retrieval_results
            )


            if not rag_context:

                response["error"] = (
                    "RAG context is empty."
                )

                return response


            # ----------------------------------------------------
            # REAL RAG ANSWER GENERATOR
            # ----------------------------------------------------

            rag_result = generate_rag_answer(
                question,
                rag_context
            )


            if not isinstance(
                rag_result,
                dict
            ):

                response["error"] = (
                    "RAG answer generator returned "
                    "an invalid response."
                )

                return response


            response["answer"] = rag_result


            response["success"] = rag_result.get(
                "success",
                False
            )


            response["error"] = rag_result.get(
                "error"
            )


        # ========================================================
        # 4. HYBRID ROUTE
        # ========================================================

        elif route == "hybrid":

            hybrid_result = run_hybrid_route(
                question
            )


            if not hybrid_result.get(
                "success",
                False
            ):

                response["error"] = hybrid_result.get(
                    "error",
                    "Hybrid processing failed."
                )

                return response


            final_result = assemble_hybrid_answer(
                question,
                hybrid_result
            )


            response["success"] = final_result.get(
                "success",
                False
            )


            response["answer"] = final_result.get(
                "answer"
            )


            response["data"] = hybrid_result[
                "sql_result"
            ].get(
                "data"
            )


            response["sql"] = hybrid_result[
                "sql_result"
            ].get(
                "sql"
            )


            response["sources"] = hybrid_result.get(
                "rag_sources",
                []
            )


            response["error"] = final_result.get(
                "error"
            )


        # ========================================================
        # 5. UNSUPPORTED
        # ========================================================

        else:

            response["error"] = (
                "This question is not supported "
                "by the current copilot."
            )


        return response


    except Exception as e:

        response["error"] = (
            f"{type(e).__name__}: {str(e)}"
        )

        return response


    finally:

        response["execution_time_ms"] = round(
            (
                time.time()
                - start_time
            ) * 1000,
            2
        )


print(
    "ask_copilot(): PASS"
)

In [0]:
# ================================================================
# CELL 10 — VERIFY ASK_COPILOT RAG BINDING
# ================================================================

print("=" * 70)
print("ASK_COPILOT → RAG FUNCTION CHECK")
print("=" * 70)


orchestrator_rag_function = (
    ask_copilot.__globals__.get(
        "generate_rag_answer"
    )
)


print(
    "Function:",
    orchestrator_rag_function
)


print()
print(
    "Source file:"
)


print(
    inspect.getsourcefile(
        orchestrator_rag_function
    )
)


print()
print(
    "Source:"
)


print(
    inspect.getsource(
        orchestrator_rag_function
    )
)


rag_source = inspect.getsource(
    orchestrator_rag_function
)


if "dummy answer" in rag_source.lower():

    raise RuntimeError(
        "ask_copilot is still using "
        "the dummy generate_rag_answer function."
    )


print()
print(
    "RAG function verification: PASS"
)

In [0]:
# ================================================================
# CELL 11 — TEST SQL ROUTE
# ================================================================

print("=" * 70)
print("TEST — SQL ROUTE")
print("=" * 70)


sql_response = ask_copilot(
    "Which region generated the highest revenue?"
)


print(
    json.dumps(
        sql_response,
        indent=2,
        default=str
    )
)


assert sql_response["success"] is True

assert sql_response["route"] == "sql"

assert sql_response["error"] is None


print()
print("SQL ROUTE: PASS")

In [0]:
# ================================================================
# CELL 12 — TEST RAG ROUTE
# ================================================================

print("=" * 70)
print("TEST — RAG ROUTE")
print("=" * 70)


rag_response = ask_copilot(
    "What is the discount policy?"
)


print(
    json.dumps(
        rag_response,
        indent=2,
        default=str
    )
)


assert rag_response["success"] is True

assert rag_response["route"] == "rag"

assert rag_response["error"] is None


rag_answer_text = str(
    rag_response["answer"]
)


assert (
    "dummy answer"
    not in rag_answer_text.lower()
)


print()
print("RAG ROUTE: PASS")
print("Real grounded RAG answer detected: YES")

In [0]:
# ================================================================
# CELL 13 — TEST HYBRID ROUTE
# ================================================================

print("=" * 70)
print("TEST — HYBRID ROUTE")
print("=" * 70)


hybrid_response = ask_copilot(
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)


print(
    json.dumps(
        hybrid_response,
        indent=2,
        default=str
    )
)


assert hybrid_response["success"] is True

assert hybrid_response["route"] == "hybrid"

assert hybrid_response["error"] is None


hybrid_answer_text = str(
    hybrid_response["answer"]
)


assert (
    "dummy answer"
    not in hybrid_answer_text.lower()
)


print()
print("HYBRID ROUTE: PASS")
print("Real grounded RAG answer detected: YES")

In [0]:
# ================================================================
# CELL 14 — FINAL ORCHESTRATOR VALIDATION
# ================================================================

print("=" * 70)
print("ORCHESTRATOR VALIDATION")
print("=" * 70)


tests = [

    (
        "SQL",
        sql_response
    ),

    (
        "RAG",
        rag_response
    ),

    (
        "HYBRID",
        hybrid_response
    )
]


successful_tests = 0


for route_name, result in tests:

    passed = (
        result.get("success") is True
        and result.get("route") is not None
        and result.get("error") is None
    )


    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )


    if passed:

        successful_tests += 1


print()
print(
    "Total tests:",
    len(tests)
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    len(tests) - successful_tests
)


if successful_tests == len(tests):

    print()
    print(
        "ORCHESTRATOR STATUS: PASS ✓"
    )

else:

    print()
    print(
        "ORCHESTRATOR STATUS: FAIL ✗"
    )